# 15. Mapping Reads to a Reference Genome

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `read_summary.py`

What the aligner decided about the first 100,000 reads.


In [ ]:
# What the aligner decided about the first 100,000 reads.
from collections import Counter

import pysam

kinds = Counter()
clipped = 0
with pysam.AlignmentFile("reads.bam") as bam:
    for number, read in enumerate(bam.fetch(until_eof=True)):
        if number == 100_000:
            break
        if read.is_secondary or read.is_supplementary:
            continue
        if read.is_unmapped:
            kinds["unmapped"] += 1
        elif read.mapping_quality < 20:
            kinds["mapped, quality below 20"] += 1
        elif read.is_proper_pair:
            kinds["mapped, properly paired"] += 1
        else:
            kinds["mapped, pair looks unusual"] += 1
        if read.cigartuples and any(
            op == 4 for op, _ in read.cigartuples
        ):
            clipped += 1

for kind, count in kinds.most_common():
    print(f"{kind:28} {count:6,}")
print(f"{'soft clipped at one end':28} {clipped:6,}")

## `coverage.py`

How deeply the reads cover each base of the REL606 genome, and which genes lie in stretches no read covers.


In [ ]:
# How deeply the reads cover each base of the REL606 genome, and which
# genes lie in stretches no read covers.
import numpy as np
import pandas as pd
import pysam

bam = pysam.AlignmentFile("reads.bam")
contig, length = bam.references[0], bam.lengths[0]

# count_coverage gives four arrays, the A, C, G and T counts at each base
depth = np.array(
    bam.count_coverage(contig, 0, length, quality_threshold=0)
).sum(axis=0)
print(f"{contig}: {length:,} bases")
print(f"  mean depth {depth.mean():.1f}, median {np.median(depth):.0f}")
print(f"  bases with no reads: {(depth == 0).sum():,}")

# Runs of zero depth: where the depth starts and stops being zero
edges = np.diff(np.concatenate([[0], depth == 0, [0]]).astype(int))
starts, ends = np.where(edges == 1)[0], np.where(edges == -1)[0]
runs = sorted(zip(ends - starts, starts, ends), reverse=True)

columns = [
    "seqid",
    "source",
    "type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes",
]
gff = pd.read_csv("rel606.gff3", sep="\t", comment="#", names=columns)
genes = gff[gff["type"] == "gene"].copy()
genes["name"] = genes["attributes"].str.extract(r"Name=([^;]+)")

print("longest stretches with no reads:")
for size, start, end in runs[:3]:
    inside = genes[(genes["start"] <= end) & (genes["end"] > start)]
    names = ", ".join(inside["name"])
    print(f"  {start + 1:,} to {end:,}, {size:,} bases")
    if names:
        print(f"    genes: {names}")

np.save("depth.npy", depth.astype(np.int32))  # for the figures